# PAE Heatmap — VPS16 × VPS33A (CORVET complex, pair CE)

Reads `CE_confidences.json` from `assembled_complex/pairs/CE/` and plots the
Predicted Aligned Error (PAE) matrix for the VPS16–VPS33A dimer, boxing the
positions of two VPS33A residues of interest, **Y438** and **I441**, together
with the VPS16 residues that surround each site in 3-D (within 8 Å in
`CE_C-CE_E.pdb`).

| Chain | Gene | Residues |
|-------|------|----------|
| C | VPS16 | 839 |
| E | VPS33A | 596 |

In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Rectangle, ConnectionPatch
from Bio.PDB import PDBParser
import warnings
warnings.filterwarnings('ignore')

In [3]:
# ── Configuration ────────────────────────────────────────────────────────────
DATA_DIR  = Path(r"N:\08_NK_structure_prediction\data\CORVET_complex\assembled_complex\pairs\CE")
CONF_FILE = DATA_DIR / "CE_confidences.json"
PDB_FILE  = DATA_DIR / "CE_C-CE_E.pdb"

# Chain → gene name mapping for this pair
CHAIN_GENE = {
    "C": "VPS16",
    "E": "VPS33A",
}

# VPS33A residues of interest: (chain, res_id, label)
MARKERS = [
    ("E", 438, "Y438"),
    ("E", 441, "I441"),
]
MARKER_COLORS = {
    "Y438": "#0072B2",
    "I441": "#D55E00",
}

# "Surrounding VPS16 residues": any chain-C residue with an atom within this
# distance (Å) of any atom of a VPS33A site residue, measured on the
# representative 3-D model (PDB_FILE).
NEARBY_CUTOFF  = 8.0
NEIGHBOR_CHAIN = "C"

# Padding (in residues) added around the VPS33A site range when defining the
# boxed / zoomed interface region.
SITE_PAD = 5

BOX_COLOR = "#222222"

FIGURE_DIR = (
    Path(r"N:\08_NK_structure_prediction\XL_MOPLC\XL_complex_structure\result")
    / "pae_vps16_vps33a"
)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Figure output → {FIGURE_DIR}")

Figure output → N:\08_NK_structure_prediction\XL_MOPLC\XL_complex_structure\result\pae_vps16_vps33a


In [4]:
# ── Nature figure style ───────────────────────────────────────────────────────
mpl.rcParams.update({
    "font.family":        "sans-serif",
    "font.sans-serif":    ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size":          7,
    "axes.titlesize":     8,
    "axes.labelsize":     7,
    "xtick.labelsize":    6,
    "ytick.labelsize":    6,
    "legend.fontsize":    6,
    "legend.frameon":     False,
    "axes.linewidth":     0.5,
    "xtick.major.width":  0.5,
    "ytick.major.width":  0.5,
    "xtick.major.size":   2.5,
    "ytick.major.size":   2.5,
    "xtick.direction":    "out",
    "ytick.direction":    "out",
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "lines.linewidth":    0.75,
    "patch.linewidth":    0.5,
    "figure.dpi":         150,
    "savefig.dpi":        300,
    "savefig.bbox":       "tight",
    "savefig.pad_inches": 0.05,
    "pdf.fonttype":       42,
    "ps.fonttype":        42,
})

NC1 = 89  / 25.4
NC2 = 183 / 25.4


def save_fig(fig, name):
    path = FIGURE_DIR / f"{name}.svg"
    fig.savefig(path, format="svg")
    print(f"Saved → {path}")

In [5]:
# ── Helper functions ──────────────────────────────────────────────────────────
def load_confidences(path):
    """Return (token_chain_ids, token_res_ids, pae_array)."""
    with open(path) as fh:
        d = json.load(fh)
    return (
        np.array(d["token_chain_ids"]),
        np.array(d["token_res_ids"], dtype=int),
        np.array(d["pae"]),
    )


def chain_boundaries(token_chains):
    """Return boundary indices where chain changes."""
    bounds = [0]
    for i in range(1, len(token_chains)):
        if token_chains[i] != token_chains[i - 1]:
            bounds.append(i)
    bounds.append(len(token_chains))
    return bounds


def compute_iptm(pae, token_chains, cutoff=12.0):
    n  = len(token_chains)
    d0 = max(1.24 * max(n - 15, 1) ** (1 / 3) - 1.8, 0.1)
    inter  = token_chains[:, None] != token_chains[None, :]
    intf   = inter & ((pae < cutoff) | (pae.T < cutoff))
    n_intf = int(intf.sum())
    if n_intf == 0:
        return np.nan
    tm = 1.0 / (1.0 + (pae / d0) ** 2)
    return float((tm * intf).sum() / n_intf)


def token_index(token_chains, token_res_ids, chain, res_id):
    """Return the token index for a given (chain, residue number)."""
    match = np.where((token_chains == chain) & (token_res_ids == res_id))[0]
    if len(match) == 0:
        raise ValueError(f"Residue {chain}{res_id} not found in token list")
    return int(match[0])


def token_span(token_chains, token_res_ids, chain, res_min, res_max):
    """Return (min_index, max_index) spanning a residue range on one chain."""
    match = np.where(
        (token_chains == chain)
        & (token_res_ids >= res_min)
        & (token_res_ids <= res_max)
    )[0]
    if len(match) == 0:
        raise ValueError(f"No residues {chain}{res_min}-{res_max} found")
    return int(match.min()), int(match.max())


def find_nearby_residues(pdb_path, site_chain, site_res_id, query_chain, cutoff):
    """
    Return sorted residue numbers on `query_chain` that have at least one
    non-hydrogen atom within `cutoff` Å of any atom of residue
    (site_chain, site_res_id) in the representative PDB model.
    """
    structure = PDBParser(QUIET=True).get_structure("s", str(pdb_path))
    model = structure[0]

    site_res = next(r for r in model[site_chain] if r.id[1] == site_res_id)
    site_atoms = np.array([a.coord for a in site_res if a.element != "H"])

    near = []
    for res in model[query_chain]:
        if res.id[0] != " ":
            continue
        res_atoms = np.array([a.coord for a in res if a.element != "H"])
        dmin = np.linalg.norm(
            site_atoms[:, None, :] - res_atoms[None, :, :], axis=-1
        ).min()
        if dmin < cutoff:
            near.append(res.id[1])
    return sorted(near)


def pick_ticks(lo, hi, n=6):
    """Return ~n unique integer tick positions evenly spaced over [lo, hi]."""
    return sorted(set(int(round(v)) for v in np.linspace(lo, hi, n)))

In [6]:
# ── Load PAE data ─────────────────────────────────────────────────────────────
tc, tr, pae = load_confidences(CONF_FILE)
N = len(tc)
bounds = chain_boundaries(tc)
iptm = compute_iptm(pae, tc)

marker_idx = [(label, token_index(tc, tr, chain, res_id))
              for chain, res_id, label in MARKERS]

print(f"Loaded PAE matrix: {pae.shape}")
for c in dict.fromkeys(tc):
    print(f"  chain {c} ({CHAIN_GENE.get(c, c)}): {(tc == c).sum()} residues")
print(f"ipTM (interface PAE < 12 Å) = {iptm:.4f}")
for label, idx in marker_idx:
    print(f"  {label} → token index {idx}")

# ── Structural neighborhood: VPS16 residues surrounding the VPS33A sites ─────
site_res_ids = [res_id for _, res_id, _ in MARKERS]
neighbor_res = set()
for chain, res_id, label in MARKERS:
    near = find_nearby_residues(PDB_FILE, chain, res_id, NEIGHBOR_CHAIN, NEARBY_CUTOFF)
    print(f"  VPS33A {label}: {len(near)} VPS16 residues within {NEARBY_CUTOFF:.0f} Å "
          f"(VPS16 {min(near)}-{max(near)})")
    neighbor_res.update(near)

site_lo, site_hi         = min(site_res_ids) - SITE_PAD, max(site_res_ids) + SITE_PAD
neighbor_lo, neighbor_hi = min(neighbor_res), max(neighbor_res)

site_span     = token_span(tc, tr, "E", site_lo, site_hi)
neighbor_span = token_span(tc, tr, NEIGHBOR_CHAIN, neighbor_lo, neighbor_hi)

print(
    f"\nCombined interface box: VPS33A {site_lo}-{site_hi}  ×  "
    f"VPS16 {neighbor_lo}-{neighbor_hi}  "
    f"({len(neighbor_res)} VPS16 residues within {NEARBY_CUTOFF:.0f} Å of Y438/I441)"
)

Loaded PAE matrix: (1435, 1435)
  chain C (VPS16): 839 residues
  chain E (VPS33A): 596 residues
ipTM (interface PAE < 12 Å) = 0.6549
  Y438 → token index 1276
  I441 → token index 1279
  VPS33A Y438: 30 VPS16 residues within 8 Å (VPS16 630-696)
  VPS33A I441: 17 VPS16 residues within 8 Å (VPS16 627-673)

Combined interface box: VPS33A 433-446  ×  VPS16 627-696  (32 VPS16 residues within 8 Å of Y438/I441)


In [ ]:
# ── Figure: overview heatmap (boxed interface) + zoomed inset ────────────────
fig, (ax1, ax2) = plt.subplots(
    1, 2, figsize=(NC2, NC2 * 0.5),
    gridspec_kw=dict(width_ratios=[1.0, 0.85], wspace=0.55),
)

# ---- Panel a: full PAE heatmap ---------------------------------------------
im = ax1.imshow(
    pae, cmap="RdYlGn_r", vmin=0, vmax=30,
    aspect="equal", interpolation="nearest", rasterized=True,
)

for b in bounds[1:-1]:
    ax1.axhline(b - 0.5, color="k", linewidth=0.6, alpha=0.9)
    ax1.axvline(b - 0.5, color="k", linewidth=0.6, alpha=0.9)

x_trans = ax1.get_xaxis_transform()   # (data, axes-fraction)
y_trans = ax1.get_yaxis_transform()   # (axes-fraction, data)

unique_chains = list(dict.fromkeys(tc))
for k, c in enumerate(unique_chains):
    mid  = (bounds[k] + bounds[k + 1]) / 2
    name = CHAIN_GENE.get(c, c)
    ax1.text(mid, -0.045, name, transform=x_trans, ha="center", va="bottom",
              fontsize=7, fontweight="bold", clip_on=False)
    ax1.text(-0.045, mid, name, transform=y_trans, ha="right", va="center",
              fontsize=7, fontweight="bold", rotation=90, clip_on=False)

# Combined box: VPS33A site region (Y438/I441 ± SITE_PAD) × surrounding
# VPS16 residues, drawn only in the bottom-left interaction block
# (rows = VPS33A site, cols = VPS16 neighbors).
site_lo_i, site_hi_i = site_span
nb_lo_i,   nb_hi_i    = neighbor_span
box_kw = dict(fill=False, edgecolor=BOX_COLOR, linewidth=1.2, zorder=6)
ax1.add_patch(Rectangle((nb_lo_i - 0.5, site_lo_i - 0.5),
                         nb_hi_i - nb_lo_i + 1, site_hi_i - site_lo_i + 1, **box_kw))

ax1.set_xticks([])
ax1.set_yticks([])
for spine in ax1.spines.values():
    spine.set_linewidth(0.5)

cb = fig.colorbar(im, ax=ax1, fraction=0.045, pad=0.03)
cb.set_label("PAE (Å)", fontsize=7)
cb.ax.tick_params(labelsize=6, width=0.5, length=2.5)
cb.outline.set_linewidth(0.5)

# ---- Panel b: zoomed sub-matrix (VPS33A site rows × VPS16 neighbor cols) ---
subpae = pae[site_lo_i:site_hi_i + 1, nb_lo_i:nb_hi_i + 1]
ax2.imshow(
    subpae, cmap="RdYlGn_r", vmin=0, vmax=30, aspect="auto",
    interpolation="nearest",
    extent=[neighbor_lo - 0.5, neighbor_hi + 0.5, site_hi + 0.5, site_lo - 0.5],
)

for chain, res_id, label in MARKERS:
    color = MARKER_COLORS[label]
    ax2.axhline(res_id, color=color, linewidth=1.0, linestyle="--", zorder=6)
    ax2.text(neighbor_hi + 1.2, res_id, label, color=color, fontsize=6.5,
              fontweight="bold", va="center", ha="left", clip_on=False)

ax2.set_xticks(pick_ticks(neighbor_lo, neighbor_hi, 5))
ax2.set_yticks(pick_ticks(site_lo, site_hi, 4))
ax2.set_xlabel("VPS16 residue", fontsize=6.5)
ax2.set_ylabel("VPS33A residue", fontsize=6.5)
ax2.tick_params(labelsize=5.5)
for spine in ax2.spines.values():
    spine.set_edgecolor(BOX_COLOR)
    spine.set_linewidth(1.2)
ax2.set_title(f"VPS33A {site_lo}–{site_hi}  ×  VPS16 {neighbor_lo}–{neighbor_hi}",
              fontsize=6.5, pad=4)

# Connector lines from the boxed region in panel a to panel b
for corner_a, corner_b in [
    ((nb_hi_i + 0.5, site_lo_i - 0.5), (0.0, 1.0)),
    ((nb_hi_i + 0.5, site_hi_i + 0.5), (0.0, 0.0)),
]:
    con = ConnectionPatch(
        xyA=corner_a, coordsA=ax1.transData,
        xyB=corner_b, coordsB=ax2.transAxes,
        color=BOX_COLOR, linewidth=0.6, linestyle=":", zorder=1,
    )
    fig.add_artist(con)

fig.suptitle(f"VPS16 × VPS33A  |  ipTM = {iptm:.3f}", y=1.12, fontsize=8)

save_fig(fig, "pae_heatmap_vps16_vps33a")
plt.show()

**Fig. 1 | PAE heatmap of the VPS16–VPS33A dimer (CORVET complex, pair CE), with the VPS33A Y438/I441 interface region boxed and zoomed.**
**(a)** Full predicted aligned error (PAE) matrix (Å) for the representative AlphaFold model of the VPS16 (chain C, 839 residues) – VPS33A (chain E, 596 residues) pair. The solid black line marks the inter-chain boundary; axes are labelled by gene name. The dark rectangle boxes the sub-matrix (rows = VPS33A, columns = VPS16) formed by the VPS33A site region (residues 433–446, i.e. Y438/I441 ± 5 residues) against the VPS16 residues with at least one atom within 8 Å of Y438 or I441 in the representative 3-D model (`CE_C-CE_E.pdb`; union of both sites, residues 627–696).
**(b)** Zoomed view of that boxed sub-matrix (dotted lines connect it to panel a), with axes labelled by real residue number; dashed lines mark the exact Y438 and I441 rows. Low PAE (green) in this patch indicates high-confidence relative positioning between the VPS33A site residues and their structural VPS16 neighbours. The colour scale (0–30 Å, green–red) is shared between both panels. The title reports the ipTM score computed from the inter-chain PAE (interface criterion: PAE < 12 Å).